In [28]:
!pip install transformers datasets accelerate -q

In [29]:
import os
import zipfile
import torch
import pandas as pd

from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BlipProcessor, BlipForConditionalGeneration
from torch.optim import AdamW

In [30]:
annotation_dir = "/kaggle/input/datasets/vishalraykhere/vrsbench/Annotations_train/Annotations_train"
image_dir = "/kaggle/input/datasets/vishalraykhere/vrsbench/Images_train/Images_train"

In [31]:
data = []

files = os.listdir(annotation_dir)

for file in tqdm(files):

    path = os.path.join(annotation_dir, file)

    with open(path) as f:
        sample = json.load(f)

    image_name = sample["image"]

    for qa in sample["qa_pairs"]:

        data.append({
            "image": image_name,
            "question": qa["question"],
            "answer": qa["answer"]
        })

df = pd.DataFrame(data)

print("Total samples:", len(df))
print(df.head())

100%|██████████| 20264/20264 [01:20<00:00, 251.38it/s]


Total samples: 85813
            image                                           question  \
0  P1439_0001.png  How many large vehicles are visible on the top...   
1  P1439_0001.png  Are there more small vehicles or large vehicle...   
2  P1439_0001.png  What is the position of the rightmost large ve...   
3  P1439_0001.png  Is there a large vehicle on the rightmost side...   
4  P1704_0031.png                    Is there a bridge in the image?   

           answer  
0               2  
1  Small vehicles  
2       Top-right  
3             Yes  
4             Yes  


In [32]:
df = df.sample(3000).reset_index(drop=True)

In [33]:
# df = df.sample(3000).reset_index(drop=True)

In [34]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

model.to(device)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

BlipForConditionalGeneration(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-0

In [42]:
class VRSDataset(Dataset):

    def __init__(self, dataframe, image_dir, processor):

        self.df = dataframe
        self.image_dir = image_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = os.path.join(self.image_dir, row["image"])
        image = Image.open(image_path).convert("RGB")

        question = row["question"]
        answer = row["answer"]

        encoding = self.processor(
            images=image,
            text=question,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = self.processor.tokenizer(
            answer,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).input_ids

        item = {
            "pixel_values": encoding["pixel_values"].squeeze(),
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": labels.squeeze()
        }

        return item

In [43]:
dataset = VRSDataset(df, image_dir, processor)

In [44]:
dataset_size = len(dataset)

client1_size = int(0.33 * dataset_size)
client2_size = int(0.33 * dataset_size)
client3_size = dataset_size - client1_size - client2_size

client1_data, client2_data, client3_data = random_split(
    dataset,
    [client1_size, client2_size, client3_size]
)

In [45]:
batch_size = 4

client1_loader = DataLoader(client1_data, batch_size=batch_size, shuffle=True)
client2_loader = DataLoader(client2_data, batch_size=batch_size, shuffle=True)
client3_loader = DataLoader(client3_data, batch_size=batch_size, shuffle=True)

In [46]:
def train_client(model, dataloader):

    optimizer = AdamW(model.parameters(), lr=5e-5)

    model.train()

    total_loss = 0

    for batch in dataloader:

        batch = {k:v.to(device) for k,v in batch.items()}

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print("Client Loss:", total_loss/len(dataloader))

    return model.state_dict()

In [47]:
def federated_average(weights):

    avg_weights = {}

    for key in weights[0].keys():

        avg_weights[key] = torch.mean(
            torch.stack([w[key] for w in weights]), dim=0
        )

    return avg_weights

In [48]:
rounds = 3

for r in range(rounds):

    print("Federated Round", r+1)

    client_weights = []

    client_weights.append(train_client(model, client1_loader))
    client_weights.append(train_client(model, client2_loader))
    client_weights.append(train_client(model, client3_loader))

    new_weights = federated_average(client_weights)

    model.load_state_dict(new_weights)

Federated Round 1
Client Loss: 1.4349828490822185
Client Loss: 0.01704119611249846
Client Loss: 0.013552251626171317
Federated Round 2


OutOfMemoryError: CUDA out of memory. Tried to allocate 240.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 53.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 13.82 GiB is allocated by PyTorch, and 564.64 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
sample = df.iloc[0]

image_path = os.path.join(image_dir, sample["image"])

image = Image.open(image_path).convert("RGB")

inputs = processor(
    images=image,
    text=sample["question"],
    return_tensors="pt"
).to(device)

output = model.generate(**inputs)

pred = processor.decode(output[0], skip_special_tokens=True)

print("Question:", sample["question"])
print("Predicted Answer:", pred)